# Generate Predictions

This notebook generates predictions using:
- Cosine similarity (text embeddings only)
- NCF baseline models (no text)
- NCF models with text embeddings (properly mapped)

## 1. Setup and Installation

In [ ]:
! pip install torch numpy tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from tqdm import tqdm
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# ============================================================
# PATHS - Update BASE_DIR to match your Google Drive structure
# ============================================================

BASE_DIR = '/content/drive/MyDrive/CIS 5300'

# Data directories (containing ncf_data/ and mappings)
DATA_DIRS = {
    'v3_5tfidf': os.path.join(BASE_DIR, 'engage_corpus_processed_v3_5tfidf'),
    'v4': os.path.join(BASE_DIR, 'engage_corpus_processed_v4'),

    # Chi-sq
    'v3_5_chi2': os.path.join(BASE_DIR, 'engage_corpus_processed_v3_5_chi2'),
    'v4_chi2': os.path.join(BASE_DIR, 'engage_corpus_processed_v4_chi2'),
}

# Embedding directories
EMBEDDING_DIRS = {
    'v3_5tfidf_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v3_5tfidf', 'filtered'),
    'v4_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v4', 'filtered'),

    # Chi-sq
    'v3_5_chi2_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v3_5_chi2', 'filtered'),
    'v4_chi2_filtered': os.path.join(BASE_DIR, 'embeddings_qwen3_8b', 'v4_chi2', 'filtered'),
}

# Model directory (from training notebook)
MODEL_DIR = os.path.join(BASE_DIR, 'trained_models')
CHECKPOINT_DIR = os.path.join(MODEL_DIR, 'checkpoints')

# Output directory for predictions
OUTPUT_DIR = os.path.join(BASE_DIR, 'predictions')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# ============================================================
# CHECKPOINT SELECTION - Choose which checkpoint to use
# ============================================================
# Options:
#   - 'best': Use the best model (lowest validation loss)
#   - 'final': Use the final model (last epoch)
#   - 'epoch_N': Use checkpoint from epoch N (e.g., 'epoch_6')
#
# In this study, we ultimately used 'best'
# ============================================================

CHECKPOINT_SELECTION = {
    'model_A': 'best',    # v3.5 + text: 'best', 'final', or 'epoch_3', 'epoch_6', etc.
    'model_B': 'best',    # v4 baseline: 'best', 'final', or 'epoch_3', 'epoch_6', etc.
    'model_C': 'best',    # v4 + text: 'best', 'final', or 'epoch_3', 'epoch_6', etc.
    'model_D': 'best',    # v3.5 TFIDF + gated fusion
    'model_E': 'best',    # v4 TFIDF + gated fusion
    'model_F': 'best',    # v4 chi2 + concat
    'model_G': 'best',    # v4 chisq + gated fusion
}

print("Checkpoint selection:")
for model, checkpoint in CHECKPOINT_SELECTION.items():
    print(f"  {model}: {checkpoint}")

In [ ]:
# List available checkpoints
print("Available checkpoints:\n")

model_names = {
    'model_A': 'model_A_v35_text',
    'model_B': 'model_B_v4_baseline',
    'model_C': 'model_C_v4_text',
    'model_D': 'model_D_tfidf_gated',
    'model_E': 'model_E_v4_tfidf_gated',
    'model_F': 'model_F_chi2_concat',
    'model_G': 'model_G_chi2_gated',
}

for short_name, full_name in model_names.items():
    print(f"{short_name} ({full_name}):")

    # Check for best/final in main directory
    best_path = os.path.join(MODEL_DIR, f'{full_name}_best.pt')
    final_path = os.path.join(MODEL_DIR, f'{full_name}_final.pt')

    if os.path.exists(best_path):
        size = os.path.getsize(best_path) / (1024*1024)
        print(f"  ✓ best ({size:.1f} MB)")
    if os.path.exists(final_path):
        size = os.path.getsize(final_path) / (1024*1024)
        print(f"  ✓ final ({size:.1f} MB)")

    # Check for epoch checkpoints
    checkpoint_subdir = os.path.join(CHECKPOINT_DIR, full_name)
    if os.path.exists(checkpoint_subdir):
        checkpoints = sorted([f for f in os.listdir(checkpoint_subdir) if f.endswith('.pt')])
        for ckpt in checkpoints:
            ckpt_path = os.path.join(checkpoint_subdir, ckpt)
            size = os.path.getsize(ckpt_path) / (1024*1024)
            print(f"  ✓ {ckpt.replace('.pt', '')} ({size:.1f} MB)")
    print()

In [ ]:
def get_model_path(short_name, checkpoint_type):
    """
    Get the full path to a model checkpoint.

    Args:
        short_name: 'model_A', 'model_B', or 'model_C'
        checkpoint_type: 'best', 'final', or 'epoch_N'

    Returns:
        Full path to the checkpoint file
    """
    full_name = model_names[short_name]

    if checkpoint_type == 'best':
        return os.path.join(MODEL_DIR, f'{full_name}_best.pt')
    elif checkpoint_type == 'final':
        return os.path.join(MODEL_DIR, f'{full_name}_final.pt')
    elif checkpoint_type.startswith('epoch_'):
        return os.path.join(CHECKPOINT_DIR, full_name, f'{checkpoint_type}.pt')
    else:
        raise ValueError(f"Unknown checkpoint type: {checkpoint_type}")


# Verify selected checkpoints exist
print("Verifying selected checkpoints...\n")
for short_name, checkpoint_type in CHECKPOINT_SELECTION.items():
    path = get_model_path(short_name, checkpoint_type)
    if os.path.exists(path):
        print(f"✓ {short_name} ({checkpoint_type}): {path}")
    else:
        print(f"✗ {short_name} ({checkpoint_type}): NOT FOUND - {path}")

## 3. Load Mappings

In [ ]:
# ============================================================
# LOAD MAPPINGS
# ============================================================
print("Loading mappings...\n")

# Load NCF data mappings (used during training)
# These define the user/subreddit indices used in the NCF model
with open(os.path.join(DATA_DIRS['v3_5tfidf'], 'user_mapping.json'), 'r') as f:
    v3_5_user_mapping = json.load(f)
with open(os.path.join(DATA_DIRS['v3_5tfidf'], 'subreddit_mapping.json'), 'r') as f:
    v3_5_sub_mapping = json.load(f)

with open(os.path.join(DATA_DIRS['v4'], 'user_mapping.json'), 'r') as f:
    v4_user_mapping = json.load(f)
with open(os.path.join(DATA_DIRS['v4'], 'subreddit_mapping.json'), 'r') as f:
    v4_sub_mapping = json.load(f)

# Chi-sq v4 data

with open(os.path.join(DATA_DIRS['v4_chi2'], 'user_mapping.json'), 'r') as f:
    v4_chi2_user_mapping = json.load(f)
with open(os.path.join(DATA_DIRS['v4_chi2'], 'subreddit_mapping.json'), 'r') as f:
    v4_chi2_sub_mapping = json.load(f)

print(f"NCF Mappings (used during model training):")
print(f"  v3.5 tfidf: {v3_5_user_mapping['num_users']} users, {v3_5_sub_mapping['num_subreddits']} subreddits")
print(f"  v4:         {v4_user_mapping['num_users']} users, {v4_sub_mapping['num_subreddits']} subreddits")
print(f"  v4_chi2:    {v4_chi2_user_mapping['num_users']} users, {v4_chi2_sub_mapping['num_subreddits']} subreddits")

# Load embedding index files (define row ordering in embedding arrays)
# These tell us which user/subreddit is at which row in the .npy files
print("\nLoading embedding index files...\n")

embedding_indices = {}

for name, emb_dir in EMBEDDING_DIRS.items():
    print(f"{name}:")

    user_ids_path = os.path.join(emb_dir, 'test', 'user_ids.json')
    sub_names_path = os.path.join(emb_dir, 'test', 'subreddit_names.json')

    if os.path.exists(user_ids_path) and os.path.exists(sub_names_path):
        with open(user_ids_path, 'r') as f:
            user_ids = json.load(f)
        with open(sub_names_path, 'r') as f:
            sub_names = json.load(f)

        embedding_indices[name] = {
            'user_ids': user_ids,
            'subreddit_names': sub_names
        }

        print(f"  Embeddings: {len(user_ids)} users, {len(sub_names)} subreddits")
    else:
        print(f"  ✗ Index files not found")
        embedding_indices[name] = None

## 4. Create Reindexing Arrays

In [ ]:
# ============================================================
# CREATE REINDEXING ARRAYS
# Maps NCF model indices → Embedding array indices
#
# This is due to the fact that the NCF data & the embeddings
# used different indexings oops
# ============================================================

def create_reindex_mapping(ncf_mapping, embedding_ids, entity_type='user'):
    num_key = 'num_users' if entity_type == 'user' else 'num_subreddits'
    num_entities = ncf_mapping[num_key]

    # Create reverse lookup: ID/name → embedding array index
    emb_to_idx = {str(eid): i for i, eid in enumerate(embedding_ids)}

    reindex_array = np.zeros(num_entities, dtype=np.int32)
    missing_mask = np.zeros(num_entities, dtype=bool)

    if entity_type == 'user':
        # For users: embedding_ids contains NCF user indices directly
        # user_ids.json = [0, 1, 2, ...] means row i has user i's embedding
        for emb_idx, ncf_user_id in enumerate(embedding_ids):
            if ncf_user_id < num_entities:
                reindex_array[ncf_user_id] = emb_idx
        # Mark users not in embedding file as missing
        embedded_users = set(embedding_ids)
        for i in range(num_entities):
            if i not in embedded_users:
                missing_mask[i] = True

    elif entity_type == 'subreddit':
        # For subreddits: embedding_ids contains names, need subreddit2idx
        subreddit2idx = ncf_mapping.get('subreddit2idx', {})
        for emb_idx, sub_name in enumerate(embedding_ids):
            if sub_name in subreddit2idx:
                ncf_idx = subreddit2idx[sub_name]
                reindex_array[ncf_idx] = emb_idx
        # Mark subreddits not in embedding file as missing
        embedded_subs = set(embedding_ids)
        for sub_name, ncf_idx in subreddit2idx.items():
            if sub_name not in embedded_subs:
                missing_mask[ncf_idx] = True

    return reindex_array, missing_mask


print("Creating reindexing arrays...\n")

reindex_mappings = {}

# v3.5 tfidf reindexing
if embedding_indices.get('v3_5tfidf_filtered'):
    print("v3.5 tfidf:")

    user_reindex, user_missing = create_reindex_mapping(
        v3_5_user_mapping,
        embedding_indices['v3_5tfidf_filtered']['user_ids'],
        'user'
    )

    sub_reindex, sub_missing = create_reindex_mapping(
        v3_5_sub_mapping,
        embedding_indices['v3_5tfidf_filtered']['subreddit_names'],
        'subreddit'
    )

    reindex_mappings['v3_5tfidf'] = {
        'user_reindex': user_reindex,
        'user_missing': user_missing,
        'sub_reindex': sub_reindex,
        'sub_missing': sub_missing
    }

    print(f"  Users: {len(user_reindex)} NCF indices → {len(embedding_indices['v3_5tfidf_filtered']['user_ids'])} embeddings")
    print(f"    Missing embeddings: {user_missing.sum()} ({100*user_missing.sum()/len(user_reindex):.1f}%)")
    print(f"  Subreddits: {len(sub_reindex)} NCF indices → {len(embedding_indices['v3_5tfidf_filtered']['subreddit_names'])} embeddings")
    print(f"    Missing embeddings: {sub_missing.sum()} ({100*sub_missing.sum()/len(sub_reindex):.1f}%)")

# v4 reindexing
if embedding_indices.get('v4_filtered'):
    print("\nv4:")

    user_reindex, user_missing = create_reindex_mapping(
        v4_user_mapping,
        embedding_indices['v4_filtered']['user_ids'],
        'user'
    )

    sub_reindex, sub_missing = create_reindex_mapping(
        v4_sub_mapping,
        embedding_indices['v4_filtered']['subreddit_names'],
        'subreddit'
    )

    reindex_mappings['v4'] = {
        'user_reindex': user_reindex,
        'user_missing': user_missing,
        'sub_reindex': sub_reindex,
        'sub_missing': sub_missing
    }

    print(f"  Users: {len(user_reindex)} NCF indices → {len(embedding_indices['v4_filtered']['user_ids'])} embeddings")
    print(f"    Missing embeddings: {user_missing.sum()} ({100*user_missing.sum()/len(user_reindex):.1f}%)")
    print(f"  Subreddits: {len(sub_reindex)} NCF indices → {len(embedding_indices['v4_filtered']['subreddit_names'])} embeddings")
    print(f"    Missing embeddings: {sub_missing.sum()} ({100*sub_missing.sum()/len(sub_reindex):.1f}%)")

# v4_chi2 reindexing
if embedding_indices.get('v4_chi2_filtered'):
    print("\nv4_chi2:")

    user_reindex, user_missing = create_reindex_mapping(
        v4_chi2_user_mapping,
        embedding_indices['v4_chi2_filtered']['user_ids'],
        'user'
    )

    sub_reindex, sub_missing = create_reindex_mapping(
        v4_chi2_sub_mapping,
        embedding_indices['v4_chi2_filtered']['subreddit_names'],
        'subreddit'
    )

    reindex_mappings['v4_chi2'] = {
        'user_reindex': user_reindex,
        'user_missing': user_missing,
        'sub_reindex': sub_reindex,
        'sub_missing': sub_missing
    }

    print(f"  Users: {len(user_reindex)} indices mapped")
    print(f"  Subreddits: {len(sub_reindex)} indices mapped")

print("\n✓ Reindexing arrays created!")

## 5. Model Definitions

In [ ]:
class NCFBaseline(nn.Module):
    """NCF Baseline Model (No Text Embeddings)"""
    def __init__(self, num_users, num_subreddits, embedding_dim=64):
        super().__init__()
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_gmf = nn.Embedding(num_subreddits, embedding_dim)
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_mlp = nn.Embedding(num_subreddits, embedding_dim)

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )
        self.final = nn.Linear(embedding_dim + 64, 1)

    def forward(self, user, subreddit):
        gmf = self.user_embedding_gmf(user) * self.sub_embedding_gmf(subreddit)
        mlp_in = torch.cat([self.user_embedding_mlp(user), self.sub_embedding_mlp(subreddit)], -1)
        mlp_out = self.mlp(mlp_in)
        return torch.sigmoid(self.final(torch.cat([gmf, mlp_out], -1))).squeeze()


class NCFWithTextEmbeddings(nn.Module):
    """NCF Model with Text Embeddings"""
    def __init__(self, num_users, num_subreddits, embedding_dim=64, text_emb_dim=128, text_proj_dim=128):
        super().__init__()
        self.user_text_proj = nn.Linear(text_emb_dim, text_proj_dim)
        self.sub_text_proj = nn.Linear(text_emb_dim, text_proj_dim)
        enhanced_dim = embedding_dim + text_proj_dim

        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_gmf = nn.Embedding(num_subreddits, embedding_dim)
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding_mlp = nn.Embedding(num_subreddits, embedding_dim)

        self.mlp = nn.Sequential(
            nn.Linear(enhanced_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )
        self.final = nn.Linear(enhanced_dim + 64, 1)

    def forward(self, user, subreddit, user_text_emb, sub_text_emb):
        user_text = self.user_text_proj(user_text_emb)
        sub_text = self.sub_text_proj(sub_text_emb)
        user_gmf = torch.cat([self.user_embedding_gmf(user), user_text], -1)
        sub_gmf = torch.cat([self.sub_embedding_gmf(subreddit), sub_text], -1)
        gmf = user_gmf * sub_gmf
        user_mlp = torch.cat([self.user_embedding_mlp(user), user_text], -1)
        sub_mlp = torch.cat([self.sub_embedding_mlp(subreddit), sub_text], -1)
        mlp_out = self.mlp(torch.cat([user_mlp, sub_mlp], -1))
        return torch.sigmoid(self.final(torch.cat([gmf, mlp_out], -1))).squeeze()


class GatedFusion(nn.Module):
    """
    Fuses an ID embedding with a Text embedding using a learnable gate.
    """
    def __init__(self, id_dim, text_dim, dropout=0.2):
        super(GatedFusion, self).__init__()
        self.text_proj = nn.Linear(text_dim, id_dim)
        self.dropout = nn.Dropout(dropout)
        self.gate_net = nn.Linear(id_dim * 2, id_dim)

    def forward(self, id_emb, text_emb):
        text_feat = F.relu(self.text_proj(text_emb))
        text_feat = self.dropout(text_feat)
        # Gate calculation
        combined = torch.cat([id_emb, text_feat], dim=1)
        gate = torch.sigmoid(self.gate_net(combined))
        # Residual fusion
        return id_emb + (gate * text_feat)

class NCFWithTextGated(nn.Module):
    """
    NCF Model that uses GatedFusion instead of simple concatenation.
    """
    def __init__(self, num_users, num_subreddits, embedding_dim=64,
                 text_emb_dim=128, text_proj_dim=128):
        super().__init__()

        # --- Gated Fusion Modules ---
        self.user_fusion = GatedFusion(embedding_dim, text_emb_dim)
        self.sub_fusion = GatedFusion(embedding_dim, text_emb_dim)

        # --- Base Embeddings ---
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.sub_embedding = nn.Embedding(num_subreddits, embedding_dim)

        # --- MLP Branch (Note: Input dim is smaller than concat model) ---
        # Because we fuse before the MLP, the input is just [User_Fused, Item_Fused]
        # Size = 64 + 64 = 128
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64)
        )

        self.final = nn.Linear(embedding_dim + 64, 1)

        # Init weights
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.sub_embedding.weight)
        # Initialize gate bias to -1 to start by trusting ID more (stability)
        nn.init.constant_(self.user_fusion.gate_net.bias, -1.0)
        nn.init.constant_(self.sub_fusion.gate_net.bias, -1.0)

    def forward(self, user, subreddit, user_text_emb, sub_text_emb):
        u_id = self.user_embedding(user)
        s_id = self.sub_embedding(subreddit)

        # Apply Gating
        u_rich = self.user_fusion(u_id, user_text_emb)
        s_rich = self.sub_fusion(s_id, sub_text_emb)

        # Standard NCF logic using the rich vectors
        gmf_out = u_rich * s_rich
        mlp_in = torch.cat([u_rich, s_rich], dim=1)
        mlp_out = self.mlp(mlp_in)

        fusion = torch.cat([gmf_out, mlp_out], dim=-1)
        return self.final(fusion).squeeze()

## 6. Prediction Functions

(With reindexing)

In [ ]:
def generate_cosine_similarity_predictions_with_reindex(
    user_embeddings, subreddit_embeddings,
    num_users_ncf, num_subs_ncf,
    user_reindex, sub_reindex
):
    """
    Generate predictions using cosine similarity with proper reindexing.

    Args:
        user_embeddings: (num_embedding_users, embedding_dim) array
        subreddit_embeddings: (num_embedding_subs, embedding_dim) array
        num_users_ncf: Number of users in NCF data
        num_subs_ncf: Number of subreddits in NCF data
        user_reindex: Array mapping NCF user index → embedding array index
        sub_reindex: Array mapping NCF subreddit index → embedding array index

    Returns:
        (num_users_ncf, num_subs_ncf) similarity matrix
    """
    # Normalize embeddings
    user_norm = user_embeddings / (np.linalg.norm(user_embeddings, axis=1, keepdims=True) + 1e-8)
    sub_norm = subreddit_embeddings / (np.linalg.norm(subreddit_embeddings, axis=1, keepdims=True) + 1e-8)

    # Reindex embeddings to match NCF ordering
    user_norm_reindexed = user_norm[user_reindex]
    sub_norm_reindexed = sub_norm[sub_reindex]

    # Compute cosine similarity
    return (user_norm_reindexed @ sub_norm_reindexed.T).astype(np.float32)

In [ ]:
def generate_ncf_predictions_with_text(model_path, user_embeddings, subreddit_embeddings,
                                        num_users, num_subreddits,
                                        user_reindex, sub_reindex,
                                        batch_size=1000,
                                        model_class=NCFWithTextEmbeddings):
    """
    Generate predictions using NCF model with text embeddings.

    Args:
        model_path: Path to model checkpoint
        user_embeddings: (num_embedding_users, text_emb_dim) array
        subreddit_embeddings: (num_embedding_subs, text_emb_dim) array
        num_users: Number of users in NCF model
        num_subreddits: Number of subreddits in NCF model
        user_reindex: Array mapping NCF user index → embedding array index
        sub_reindex: Array mapping NCF subreddit index → embedding array index
        batch_size: Batch size for prediction

    Returns:
        (num_users, num_subreddits) prediction matrix
    """
    # Initialize model
    model = model_class(num_users, num_subreddits).to(device)
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"  Loaded from epoch {checkpoint.get('epoch', '?')} "
              f"(val_loss: {checkpoint.get('val_loss', '?'):.4f})")
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    # Prepare embeddings on device
    predictions = np.zeros((num_users, num_subreddits), dtype=np.float32)
    user_emb_tensor = torch.FloatTensor(user_embeddings).to(device)
    sub_emb_tensor = torch.FloatTensor(subreddit_embeddings).to(device)

    # Convert reindex arrays to tensors
    user_reindex_tensor = torch.LongTensor(user_reindex).to(device)
    sub_reindex_tensor = torch.LongTensor(sub_reindex).to(device)

    # Generate predictions
    with torch.no_grad():
        for user_id in tqdm(range(num_users), desc="  Generating predictions"):
            for start in range(0, num_subreddits, batch_size):
                end = min(start + batch_size, num_subreddits)
                batch_size_actual = end - start

                # NCF model indices
                user_tensor = torch.LongTensor([user_id] * batch_size_actual).to(device)
                sub_tensor = torch.LongTensor(list(range(start, end))).to(device)

                # CRITICAL: Use reindexing to get correct embeddings
                user_emb_idx = user_reindex_tensor[user_id]
                sub_emb_indices = sub_reindex_tensor[start:end]

                user_emb_batch = user_emb_tensor[user_emb_idx].unsqueeze(0).expand(batch_size_actual, -1)
                sub_emb_batch = sub_emb_tensor[sub_emb_indices]

                scores = model(user_tensor, sub_tensor, user_emb_batch, sub_emb_batch)
                predictions[user_id, start:end] = scores.cpu().numpy()

    return predictions

In [ ]:
def generate_ncf_baseline_predictions(model_path, num_users, num_subreddits, batch_size=1000):
    """
    Generate predictions using NCF baseline (no text).

    Args:
        model_path: Path to model checkpoint
        num_users: Number of users
        num_subreddits: Number of subreddits
        batch_size: Batch size for prediction

    Returns:
        (num_users, num_subreddits) prediction matrix
    """
    # Initialize model
    model = NCFBaseline(num_users, num_subreddits).to(device)

    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"  Loaded from epoch {checkpoint.get('epoch', '?')} "
              f"(val_loss: {checkpoint.get('val_loss', '?'):.4f})")
    else:
        model.load_state_dict(checkpoint)

    model.eval()

    # Generate predictions
    predictions = np.zeros((num_users, num_subreddits), dtype=np.float32)

    with torch.no_grad():
        for user_id in tqdm(range(num_users), desc="  Generating predictions"):
            for start in range(0, num_subreddits, batch_size):
                end = min(start + batch_size, num_subreddits)
                batch_size_actual = end - start

                user_tensor = torch.LongTensor([user_id] * batch_size_actual).to(device)
                sub_tensor = torch.LongTensor(list(range(start, end))).to(device)

                scores = model(user_tensor, sub_tensor)
                predictions[user_id, start:end] = scores.cpu().numpy()

    return predictions

## 7. Generate Cosine Similarity Predictions

In [ ]:
print("="*70)
print("GENERATING COSINE SIMILARITY PREDICTIONS (WITH REINDEXING)")
print("="*70)

cosine_configs = [
    ('cosine_v35_128', 'v3_5tfidf_filtered', 128, 'v3_5tfidf'),
    ('cosine_v35_4096', 'v3_5tfidf_filtered', 4096, 'v3_5tfidf'),
    ('cosine_v4_128', 'v4_filtered', 128, 'v4'),
    ('cosine_v4_4096', 'v4_filtered', 4096, 'v4'),
    ('cosine_v4_chi2_128', 'v4_chi2_filtered', 128, 'v4_chi2'),
    ('cosine_v4_chi2_4096', 'v4_chi2_filtered', 4096, 'v4')
]

cosine_results = {}

for pred_name, emb_config, dim, data_version in cosine_configs:
    print(f"\n{pred_name}: {emb_config} | {dim}-dim")

    # Check if we have reindex mappings
    if data_version not in reindex_mappings:
        print(f"  ✗ No reindexing available for {data_version}")
        continue

    # Load embeddings
    user_emb_path = os.path.join(EMBEDDING_DIRS[emb_config], 'test', f'user_embeddings_{dim}.npy')
    sub_emb_path = os.path.join(EMBEDDING_DIRS[emb_config], 'test', f'subreddit_embeddings_{dim}.npy')

    if not os.path.exists(user_emb_path):
        print(f"  ✗ User embeddings not found: {user_emb_path}")
        continue
    if not os.path.exists(sub_emb_path):
        print(f"  ✗ Subreddit embeddings not found: {sub_emb_path}")
        continue

    user_emb = np.load(user_emb_path)
    sub_emb = np.load(sub_emb_path)

    print(f"  User embeddings: {user_emb.shape}")
    print(f"  Subreddit embeddings: {sub_emb.shape}")

    # Get reindex arrays and NCF dimensions
    mapping = v3_5_user_mapping if data_version == 'v3_5tfidf' else v4_user_mapping
    sub_mapping = v3_5_sub_mapping if data_version == 'v3_5tfidf' else v4_sub_mapping

    # Generate predictions with reindexing
    predictions = generate_cosine_similarity_predictions_with_reindex(
        user_emb, sub_emb,
        mapping['num_users'], sub_mapping['num_subreddits'],
        reindex_mappings[data_version]['user_reindex'],
        reindex_mappings[data_version]['sub_reindex']
    )

    # Save
    output_path = os.path.join(OUTPUT_DIR, f'{pred_name}.npy')
    np.save(output_path, predictions)

    cosine_results[pred_name] = {
        'shape': predictions.shape,
        'data_version': data_version,
        'dim': dim
    }

    print(f"  ✓ Saved: {output_path} | Shape: {predictions.shape}")

## 8. Generate NCF Model Predictions

In [ ]:
print("\n" + "="*70)
print("GENERATING NCF MODEL PREDICTIONS")
print("="*70)

ncf_results = {}

In [ ]:
# Model A: v3.5 tfidf + Text Embeddings
print("\n" + "-"*50)
print("Model A: v3.5 tfidf + Text Embeddings")
print("-"*50)

model_path = get_model_path('model_A', CHECKPOINT_SELECTION['model_A'])
print(f"  Checkpoint: {CHECKPOINT_SELECTION['model_A']}")
print(f"  Path: {model_path}")

# Load embeddings
user_emb = np.load(os.path.join(EMBEDDING_DIRS['v3_5tfidf_filtered'], 'test', 'user_embeddings_128.npy'))
sub_emb = np.load(os.path.join(EMBEDDING_DIRS['v3_5tfidf_filtered'], 'test', 'subreddit_embeddings_128.npy'))

print(f"  User embeddings: {user_emb.shape}")
print(f"  Subreddit embeddings: {sub_emb.shape}")

# Generate predictions with reindexing
predictions_A = generate_ncf_predictions_with_text(
    model_path, user_emb, sub_emb,
    v3_5_user_mapping['num_users'], v3_5_sub_mapping['num_subreddits'],
    reindex_mappings['v3_5tfidf']['user_reindex'],
    reindex_mappings['v3_5tfidf']['sub_reindex']
)

# Save
output_path = os.path.join(OUTPUT_DIR, 'predictions_model_A.npy')
np.save(output_path, predictions_A)

ncf_results['model_A'] = {
    'shape': predictions_A.shape,
    'data_version': 'v3_5',
    'checkpoint': CHECKPOINT_SELECTION['model_A']
}

print(f"  ✓ Saved: {output_path} | Shape: {predictions_A.shape}")

In [ ]:
# Model B: v4 Baseline (No Text)
print("\n" + "-"*50)
print("Model B: v4 Baseline (No Text)")
print("-"*50)

model_path = get_model_path('model_B', CHECKPOINT_SELECTION['model_B'])
print(f"  Checkpoint: {CHECKPOINT_SELECTION['model_B']}")
print(f"  Path: {model_path}")

# Generate predictions (no embeddings needed for baseline)
predictions_B = generate_ncf_baseline_predictions(
    model_path,
    v4_user_mapping['num_users'], v4_sub_mapping['num_subreddits']
)

# Save
output_path = os.path.join(OUTPUT_DIR, 'predictions_model_B.npy')
np.save(output_path, predictions_B)

ncf_results['model_B'] = {
    'shape': predictions_B.shape,
    'data_version': 'v4',
    'checkpoint': CHECKPOINT_SELECTION['model_B']
}

print(f"  ✓ Saved: {output_path} | Shape: {predictions_B.shape}")

In [ ]:
# Model C: v4 + Text Embeddings
print("\n" + "-"*50)
print("Model C: v4 + Text Embeddings")
print("-"*50)

model_path = get_model_path('model_C', CHECKPOINT_SELECTION['model_C'])
print(f"  Checkpoint: {CHECKPOINT_SELECTION['model_C']}")
print(f"  Path: {model_path}")

# Load embeddings
user_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_filtered'], 'test', 'user_embeddings_128.npy'))
sub_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_filtered'], 'test', 'subreddit_embeddings_128.npy'))

print(f"  User embeddings: {user_emb.shape}")
print(f"  Subreddit embeddings: {sub_emb.shape}")

# Generate predictions with reindexing
predictions_C = generate_ncf_predictions_with_text(
    model_path, user_emb, sub_emb,
    v4_user_mapping['num_users'], v4_sub_mapping['num_subreddits'],
    reindex_mappings['v4']['user_reindex'],
    reindex_mappings['v4']['sub_reindex']
)

# Save
output_path = os.path.join(OUTPUT_DIR, 'predictions_model_C.npy')
np.save(output_path, predictions_C)

ncf_results['model_C'] = {
    'shape': predictions_C.shape,
    'data_version': 'v4',
    'checkpoint': CHECKPOINT_SELECTION['model_C']
}

print(f"  ✓ Saved: {output_path} | Shape: {predictions_C.shape}")

In [ ]:
# Model D: TF-IDF + Gated Fusion (uses v3.5 data)
print("\n" + "-"*50)
print("Model D: TF-IDF + Gated Fusion")
print("-"*50)

if 'model_D' in CHECKPOINT_SELECTION:
    model_path = get_model_path('model_D', CHECKPOINT_SELECTION['model_D'])

    # Load v3.5 embeddings
    user_emb = np.load(os.path.join(EMBEDDING_DIRS['v3_5tfidf_filtered'], 'test', 'user_embeddings_128.npy'))
    sub_emb = np.load(os.path.join(EMBEDDING_DIRS['v3_5tfidf_filtered'], 'test', 'subreddit_embeddings_128.npy'))

    predictions_D = generate_ncf_predictions_with_text(
        model_path, user_emb, sub_emb,
        v3_5_user_mapping['num_users'], v3_5_sub_mapping['num_subreddits'],
        reindex_mappings['v3_5tfidf']['user_reindex'],
        reindex_mappings['v3_5tfidf']['sub_reindex'],
        model_class=NCFWithTextGated # Gated
    )
    np.save(os.path.join(OUTPUT_DIR, 'predictions_model_D.npy'), predictions_D)
    ncf_results['model_D'] = {'shape': predictions_D.shape, 'data': 'v3.5', 'type': 'Gated'}
    print(f"  ✓ Saved Model D")

In [ ]:
# Model E: v4 TF-IDF + Gated Fusion (uses v4 data)
print("\n" + "-"*50)
print("Model E: v4 TF-IDF + Gated Fusion")
print("-"*50)

if 'model_E' in CHECKPOINT_SELECTION:
    model_path = get_model_path('model_E', CHECKPOINT_SELECTION['model_E'])

    # Load v4 embeddings
    user_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_filtered'], 'test', 'user_embeddings_128.npy'))
    sub_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_filtered'], 'test', 'subreddit_embeddings_128.npy'))

    predictions_E = generate_ncf_predictions_with_text(
        model_path, user_emb, sub_emb,
        v4_user_mapping['num_users'], v4_sub_mapping['num_subreddits'],
        reindex_mappings['v4']['user_reindex'],
        reindex_mappings['v4']['sub_reindex'],
        model_class=NCFWithTextGated # Gated
    )
    np.save(os.path.join(OUTPUT_DIR, 'predictions_model_E.npy'), predictions_E)
    ncf_results['model_E'] = {'shape': predictions_E.shape, 'data': 'v4', 'type': 'Gated'}
    print(f"  ✓ Saved Model E")

In [ ]:
# Model F: Chi-Square + Non-Gated (uses v4_chi2 data)
print("\n" + "-"*50)
print("Model F: Chi-Square + Concat (Non-Gated)")
print("-"*50)

if 'model_F' in CHECKPOINT_SELECTION:
    model_path = get_model_path('model_F', CHECKPOINT_SELECTION['model_F'])

    # Load v4_chi2 embeddings
    user_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_chi2_filtered'], 'test', 'user_embeddings_128.npy'))
    sub_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_chi2_filtered'], 'test', 'subreddit_embeddings_128.npy'))

    predictions_F = generate_ncf_predictions_with_text(
        model_path, user_emb, sub_emb,
        v4_chi2_user_mapping['num_users'], v4_chi2_sub_mapping['num_subreddits'],
        reindex_mappings['v4_chi2']['user_reindex'],
        reindex_mappings['v4_chi2']['sub_reindex'],
        model_class=NCFWithTextEmbeddings # Standard concat
    )
    np.save(os.path.join(OUTPUT_DIR, 'predictions_model_F.npy'), predictions_F)
    ncf_results['model_F'] = {'shape': predictions_F.shape, 'data': 'v4_chi2', 'type': 'Concat'}
    print(f"  ✓ Saved Model F")


In [ ]:
# Model G: Chi-Square + Gated Fusion (uses v4_chi2 data)
print("\n" + "-"*50)
print("Model G: Chi-Square + Gated Fusion")
print("-"*50)

if 'model_G' in CHECKPOINT_SELECTION:
    model_path = get_model_path('model_G', CHECKPOINT_SELECTION['model_G'])

    # Load v4_chi2 embeddings (Reuse from F, but good to be explicit)
    user_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_chi2_filtered'], 'test', 'user_embeddings_128.npy'))
    sub_emb = np.load(os.path.join(EMBEDDING_DIRS['v4_chi2_filtered'], 'test', 'subreddit_embeddings_128.npy'))

    predictions_G = generate_ncf_predictions_with_text(
        model_path, user_emb, sub_emb,
        v4_chi2_user_mapping['num_users'], v4_chi2_sub_mapping['num_subreddits'],
        reindex_mappings['v4_chi2']['user_reindex'],
        reindex_mappings['v4_chi2']['sub_reindex'],
        model_class=NCFWithTextGated # Gated
    )
    np.save(os.path.join(OUTPUT_DIR, 'predictions_model_G.npy'), predictions_G)
    ncf_results['model_G'] = {'shape': predictions_G.shape, 'data': 'v4_chi2', 'type': 'Gated'}
    print(f"  ✓ Saved Model G")

## 9. Generate Ground Truth Files

In [ ]:
print("\n" + "="*70)
print("GENERATING GROUND TRUTH FILES")
print("="*70)

In [ ]:
# Ground truth for v3.5 (single-label: array of subreddit indices)
print("\nGenerating ground truth for v3.5 (single-label)...")

ground_truth_v3_5 = []
test_file_v35 = os.path.join(DATA_DIRS['v3_5tfidf'], 'ncf_data', 'test.tsv')

with open(test_file_v35, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            ground_truth_v3_5.append(int(parts[1]))

ground_truth_v3_5 = np.array(ground_truth_v3_5, dtype=np.int32)
np.save(os.path.join(OUTPUT_DIR, 'ground_truth_v35_test.npy'), ground_truth_v3_5)

print(f"  ✓ Saved: ground_truth_v35_test.npy | Shape: {ground_truth_v3_5.shape}")
print(f"  Unique subreddits: {len(np.unique(ground_truth_v3_5))}")

In [ ]:
# Ground truth for v4 (multi-label: binary matrix)
print("\nGenerating ground truth for v4 (multi-label)...")

num_users_v4 = v4_user_mapping['num_users']
num_subs_v4 = v4_sub_mapping['num_subreddits']
ground_truth_v4 = np.zeros((num_users_v4, num_subs_v4), dtype=np.float32)

test_file_v4 = os.path.join(DATA_DIRS['v4'], 'ncf_data', 'test.tsv')
interaction_count = 0

with open(test_file_v4, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 2:
            user_id = int(parts[0])
            sub_id = int(parts[1])
            if user_id < num_users_v4 and sub_id < num_subs_v4:
                ground_truth_v4[user_id, sub_id] = 1
                interaction_count += 1

np.save(os.path.join(OUTPUT_DIR, 'ground_truth_v4_test.npy'), ground_truth_v4)

print(f"  ✓ Saved: ground_truth_v4_test.npy | Shape: {ground_truth_v4.shape}")
print(f"  Total positive interactions: {interaction_count}")
print(f"  Matrix sum (verification): {ground_truth_v4.sum():.0f}")

In [ ]:
# Ground truth for v4_chi2 (multi-label)
# Models F and G use these specific indices
print("\nGenerating ground truth for v4_chi2 (multi-label)...")

num_users_chi2 = v4_chi2_user_mapping['num_users']
num_subs_chi2 = v4_chi2_sub_mapping['num_subreddits']
ground_truth_chi2 = np.zeros((num_users_chi2, num_subs_chi2), dtype=np.float32)

test_file_chi2 = os.path.join(DATA_DIRS['v4_chi2'], 'ncf_data', 'test.tsv')
interaction_count = 0

if os.path.exists(test_file_chi2):
    with open(test_file_chi2, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                user_id = int(parts[0])
                sub_id = int(parts[1])
                if user_id < num_users_chi2 and sub_id < num_subs_chi2:
                    ground_truth_chi2[user_id, sub_id] = 1
                    interaction_count += 1

    np.save(os.path.join(OUTPUT_DIR, 'ground_truth_v4_chi2_test.npy'), ground_truth_chi2)
    print(f"  ✓ Saved: ground_truth_v4_chi2_test.npy | Shape: {ground_truth_chi2.shape}")
    print(f"  Total positive interactions: {interaction_count}")
else:
    print(f"  ✗ Test file not found: {test_file_chi2}")

## 10. Summary

In [ ]:
print("\n" + "="*70)
print("ALL PREDICTIONS GENERATED!")
print("="*70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print("\n" + "-"*50)
print("COSINE SIMILARITY PREDICTIONS")
print("-"*50)
for name, info in cosine_results.items():
    print(f"  {name}: {info['shape']} ({info['data_version']}, {info['dim']}-dim)")

print("\n" + "-"*50)
print("NCF MODEL PREDICTIONS")
print("-"*50)
for name, info in ncf_results.items():
    print(f"  {name}: {info['shape']} ({info['data_version']}, checkpoint: {info['checkpoint']})")

print("\n" + "-"*50)
print("ALL FILES")
print("-"*50)
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# Create a summary JSON file
summary = {
    'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'cosine_predictions': cosine_results,
    'ncf_predictions': ncf_results,
    'checkpoint_selection': CHECKPOINT_SELECTION,
    'reindexing_info': {
        'v3_5tfidf': {
            'user_missing_count': int(reindex_mappings['v3_5tfidf']['user_missing'].sum()),
            'sub_missing_count': int(reindex_mappings['v3_5tfidf']['sub_missing'].sum())
        } if 'v3_5tfidf' in reindex_mappings else None,
        'v4': {
            'user_missing_count': int(reindex_mappings['v4']['user_missing'].sum()),
            'sub_missing_count': int(reindex_mappings['v4']['sub_missing'].sum())
        } if 'v4' in reindex_mappings else None
    },
    'files': sorted(os.listdir(OUTPUT_DIR))
}

with open(os.path.join(OUTPUT_DIR, 'prediction_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print("\nSaved prediction_summary.json")